In [ ]:
import pyspark

In [1]:
from pyspark.sql import SparkSession, types

In [2]:
# object for interacting with spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/11 14:45:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#save it with data/pq/yellow/2024/10/yellow_tripdata_2024-10.parquet

In [4]:
df_yellow_1024 = spark.read \
        .option("header","true") \
        .parquet('data/pq/yellow/2024/10/yellow_tripdata_2024-10.parquet')

In [19]:
df_yellow_1024.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [20]:
df_yellow_1024 \
    .repartition(4) \
    .write.parquet('data/pq/yellow/2024/11/')

In [5]:
df_yellow_1024.registerTempTable('yellow_trips_data')

/home/vologzhaninsergey23/anaconda3/envs/jupyter_env/lib/python3.10/site-packages/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [25]:
#df_yellow_1024.show()

In [26]:
# Question 3: Count records. How many taxi trips were there on the 15th of October? Consider only trips that started on the 15th of October.
spark.sql("""
SELECT DATE(tpep_pickup_datetime), COUNT(1) 
FROM yellow_trips_data 
WHERE DATE(tpep_pickup_datetime) = '2024-10-15'
GROUP BY 1
""").show()

[Stage 1:>                                                          (0 + 4) / 4]

+--------------------+--------+
|tpep_pickup_datetime|count(1)|
+--------------------+--------+
|          2024-10-15|  128893|
+--------------------+--------+



In [40]:
# Question 4: Longest trip. What is the length of the longest trip in the dataset in hours?
spark.sql("""
SELECT (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600.0 AS hours_diff
FROM yellow_trips_data
ORDER BY 1 DESC
LIMIT 1
""").show()

[Stage 19:===========================================>              (3 + 1) / 4]

+----------+
|hours_diff|
+----------+
|162.617778|
+----------+



In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-11 14:45:51--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.46, 18.155.128.222, 18.155.128.187, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.46|:443... connected.
200 OKequest sent, awaiting response... 
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-11 14:45:51 (172 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [7]:
import pandas as pd

In [8]:
df_taxi_zone_lookup_pd = pd.read_csv('taxi_zone_lookup.csv', nrows=1000)

In [14]:
df_taxi_zone_lookup_pd['Zone'] = df_taxi_zone_lookup_pd['Zone'].astype(str)

In [17]:
df_taxi_zone_lookup_pd['service_zone'] = df_taxi_zone_lookup_pd['service_zone'].astype(str)

In [20]:
df_taxi_zone_lookup_pd['Borough'] = df_taxi_zone_lookup_pd['Borough'].astype(str)

In [42]:
df_taxi_zone_lookup_pd

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,nan,nan


In [22]:
spark.createDataFrame(df_taxi_zone_lookup_pd).schema

/home/vologzhaninsergey23/anaconda3/envs/jupyter_env/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:474: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
/home/vologzhaninsergey23/anaconda3/envs/jupyter_env/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


StructType([StructField('LocationID', LongType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [23]:
taxi_zone_schema = types.StructType([
    types.StructField("LocationID", types.IntegerType(), True),
    types.StructField("Borough", types.TimestampType(), True),
    types.StructField("Zone", types.TimestampType(), True),
    types.StructField("service_zone", types.StringType(), True)
])

In [46]:
df_taxi_zone_lookup = spark.read \
    .option("header","true") \
    .csv('taxi_zone_lookup.csv')

In [48]:
df_taxi_zone_lookup.registerTempTable('taxi_zone_lookup')

/home/vologzhaninsergey23/anaconda3/envs/jupyter_env/lib/python3.10/site-packages/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [51]:
#Question 6: Least frequent pickup location zone
spark.sql("""
SELECT trips_data.PULocationID, zone.Zone, COUNT(1)
FROM yellow_trips_data AS trips_data
LEFT JOIN taxi_zone_lookup AS zone
       ON trips_data.PULocationID = zone.LocationID
GROUP BY 1,2
ORDER BY 3
""").show()

+------------+--------------------+--------+
|PULocationID|                Zone|count(1)|
+------------+--------------------+--------+
|         105|Governor's Island...|       1|
|         199|       Rikers Island|       2|
|           5|       Arden Heights|       2|
|           2|         Jamaica Bay|       3|
|         111| Green-Wood Cemetery|       3|
|         204|   Rossville/Woodrow|       4|
|         187|       Port Richmond|       4|
|         245|       West Brighton|       4|
|          44|Charleston/Totten...|       4|
|          84|Eltingville/Annad...|       4|
|         109|         Great Kills|       6|
|          59|        Crotona Park|       6|
|         156|     Mariners Harbor|       7|
|         118|Heartland Village...|       7|
|         176|             Oakwood|       9|
|         206|Saint George/New ...|       9|
|          30|       Broad Channel|      10|
|         172|New Dorp/Midland ...|      10|
|         184|     Pelham Bay Park|      12|
|         